# Model 03 — Customer Lifetime Value Prediction
## 02. Feature Engineering & CLV Definition

### Objective
Define a **one-year future purchase-value proxy**, build customer-level features using only the observation period, and assess target quality before modelling.

The selected time windows are:
- Observation window: 1 Dec 2009 to 30 Nov 2010 (end-exclusive: 1 Dec 2010)
- Prediction window: 1 Dec 2010 to 30 Nov 2011 (end-exclusive: 1 Dec 2011)

The dataset extends to 9 Dec 2011; the final nine days are outside the selected prediction window.

**Primary target:** `future_gross_purchase_value`, calculated as `Quantity × Price` for qualifying positive, non-cancelled purchase lines in the prediction window. This is a one-year future purchase-value proxy—not profit and not true lifetime value. The net-transaction measure is retained for diagnostic comparison only.

**Leakage rule:** features may use observation-window records only; the target may use prediction-window records only.


## 1. Imports and paths

Run this notebook from `ml/model_03_customer_lifetime_value/notebooks/`, consistent with Notebook 01.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

MODEL_ROOT = Path.cwd().parent
DATA_PATH = MODEL_ROOT / "data" / "raw" / "online_retail_II.xlsx"

print("Model root:", MODEL_ROOT)
print("Dataset path:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Run the notebook from the model's notebooks directory or correct DATA_PATH."
    )

Model root: d:\MCA PU\Semesters\Semester 3\MCA_Projects\the-enterprise-ai-decision-intelligence-platform\ml\model_03_customer_lifetime_value
Dataset path: d:\MCA PU\Semesters\Semester 3\MCA_Projects\the-enterprise-ai-decision-intelligence-platform\ml\model_03_customer_lifetime_value\data\raw\online_retail_II.xlsx
Dataset exists: True


## 2. Load and validate the transaction data

In [2]:
excel_file = pd.ExcelFile(DATA_PATH)
raw_frames = [
    pd.read_excel(excel_file, sheet_name=sheet)
    for sheet in excel_file.sheet_names
]
df = pd.concat(raw_frames, ignore_index=True)

required_columns = [
    "Invoice", "StockCode", "Description", "Quantity",
    "InvoiceDate", "Price", "Customer ID", "Country"
]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise KeyError(
        f"Expected columns missing: {missing_columns}. "
        f"Available columns: {df.columns.tolist()}"
    )

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["TransactionRevenue"] = df["Quantity"] * df["Price"]

print("Rows:", f"{len(df):,}")
print("Date range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())
print("Missing customer IDs:", f"{df['Customer ID'].isna().sum():,}")
df.head()

Rows: 1,067,371
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Missing customer IDs: 243,007


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TransactionRevenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom,83.40
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,81.00
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,81.00
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom,100.80
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom,30.00


## 3. Candidate temporal windows

Use a full year for historical behaviour and the following full year for the target. These dates fit the available dataset coverage, but the tables below must be reviewed before accepting this design.

In [3]:
OBS_START = pd.Timestamp("2009-12-01")
OBS_END = pd.Timestamp("2010-12-01")   # exclusive
TARGET_START = OBS_END
TARGET_END = pd.Timestamp("2011-12-01")  # exclusive

assert OBS_START < OBS_END < TARGET_END
assert df["InvoiceDate"].notna().any()

date_coverage = pd.DataFrame({
    "period": ["Observation", "Prediction target"],
    "start_inclusive": [OBS_START, TARGET_START],
    "end_exclusive": [OBS_END, TARGET_END],
    "calendar_days": [(OBS_END - OBS_START).days, (TARGET_END - TARGET_START).days],
})
display(date_coverage)

monthly_coverage = (
    df.dropna(subset=["InvoiceDate"])
      .assign(month=df["InvoiceDate"].dt.to_period("M").astype(str))
      .groupby("month")
      .agg(
          transaction_rows=("Invoice", "size"),
          identified_customers=("Customer ID", "nunique"),
          distinct_invoices=("Invoice", "nunique"),
      )
      .reset_index()
)
display(monthly_coverage)

,period,start_inclusive,end_exclusive,calendar_days
0,Observation,2009-12-01,2010-12-01,365
1,Prediction target,2010-12-01,2011-12-01,365


,month,transaction_rows,identified_customers,distinct_invoices
0,2009-12,45228,1045,2330
1,2010-01,31555,786,1633
2,2010-02,29388,807,1969
3,2010-03,41511,1111,2367
4,2010-04,34057,998,1892
5,2010-05,35323,1062,2418
6,2010-06,39983,1095,2216
7,2010-07,33383,988,2017
8,2010-08,33306,964,1877
9,2010-09,42091,1202,2375


## 4. Create separate observation and target records

Customer IDs and dates are required for customer-level modelling. Keep the raw `df` unchanged. The positive-purchase subset follows Notebook 01's provisional exploratory rule; we will also compare a net-revenue target that retains negative/return lines to understand the impact of that decision.

In [4]:
customer_date_records = df.dropna(
    subset=["Customer ID", "InvoiceDate"]
).copy()

not_cancelled = ~customer_date_records["Invoice"].astype("string").str.startswith("C", na=False)
positive_purchase_mask = (
    not_cancelled
    & customer_date_records["Quantity"].gt(0)
    & customer_date_records["Price"].gt(0)
)

observation_all = customer_date_records.loc[
    customer_date_records["InvoiceDate"].ge(OBS_START)
    & customer_date_records["InvoiceDate"].lt(OBS_END)
].copy()

observation_purchases = observation_all.loc[
    positive_purchase_mask.loc[observation_all.index]
].copy()

target_all = customer_date_records.loc[
    customer_date_records["InvoiceDate"].ge(TARGET_START)
    & customer_date_records["InvoiceDate"].lt(TARGET_END)
].copy()

target_purchases = target_all.loc[
    positive_purchase_mask.loc[target_all.index]
].copy()

print("Observation rows (all identified-customer transactions):", f"{len(observation_all):,}")
print("Observation rows (provisional purchases):", f"{len(observation_purchases):,}")
print("Target rows (all identified-customer transactions):", f"{len(target_all):,}")
print("Target rows (provisional purchases):", f"{len(target_purchases):,}")

Observation rows (all identified-customer transactions): 402,731
Observation rows (provisional purchases): 393,160
Target rows (all identified-customer transactions): 403,971
Target rows (provisional purchases): 395,084


## 5. Build observation-window customer features

All features below are computed strictly before `OBS_END`. These are candidate features; their usefulness and stability will be evaluated later.

In [5]:
obs = observation_purchases.copy()
obs["line_revenue"] = obs["Quantity"] * obs["Price"]

# Aggregate first to invoice level so invoice-level statistics are not distorted
# by the number of product lines in an invoice.
obs_invoice = (
    obs.groupby(["Customer ID", "Invoice"], as_index=False)
       .agg(
           invoice_date=("InvoiceDate", "min"),
           invoice_value=("line_revenue", "sum"),
           item_quantity=("Quantity", "sum"),
           product_line_count=("StockCode", "size"),
       )
)

feature_agg = (
    obs_invoice.groupby("Customer ID")
       .agg(
           observation_invoice_count=("Invoice", "nunique"),
           observation_total_value=("invoice_value", "sum"),
           observation_average_invoice_value=("invoice_value", "mean"),
           observation_median_invoice_value=("invoice_value", "median"),
           observation_total_quantity=("item_quantity", "sum"),
           observation_product_line_count=("product_line_count", "sum"),
           first_purchase_date=("invoice_date", "min"),
           last_purchase_date=("invoice_date", "max"),
       )
       .reset_index()
)

feature_agg["recency_days"] = (
    OBS_END - feature_agg["last_purchase_date"]
).dt.days
feature_agg["customer_tenure_days"] = (
    feature_agg["last_purchase_date"] - feature_agg["first_purchase_date"]
).dt.days
feature_agg["average_days_between_observed_purchases"] = np.where(
    feature_agg["observation_invoice_count"] > 1,
    feature_agg["customer_tenure_days"] / (feature_agg["observation_invoice_count"] - 1),
    np.nan,
)

print("Customers with at least one qualifying observation-period purchase:",
      f"{len(feature_agg):,}")
display(feature_agg.head())
display(feature_agg.describe(include="all").T)

Customers with at least one qualifying observation-period purchase: 4,266


,Customer ID,observation_invoice_count,observation_total_value,observation_average_invoice_value,observation_median_invoice_value,observation_total_quantity,observation_product_line_count,first_purchase_date,last_purchase_date,recency_days,customer_tenure_days,average_days_between_observed_purchases
0,"12,346.00",11,372.86,33.90,22.50,70,33,2009-12-14 08:34:00,2010-06-28 13:53:00,155,196,19.60
1,"12,347.00",1,611.53,611.53,611.53,509,40,2010-10-31 14:20:00,2010-10-31 14:20:00,30,0,NaN
2,"12,348.00",1,222.16,222.16,222.16,373,20,2010-09-27 14:59:00,2010-09-27 14:59:00,64,0,NaN
3,"12,349.00",3,"2,671.14",890.38,"1,068.52",993,102,2010-04-29 13:20:00,2010-10-28 08:23:00,33,181,90.50
4,"12,351.00",1,300.93,300.93,300.93,261,21,2010-11-29 15:23:00,2010-11-29 15:23:00,1,0,NaN


,count,mean,min,25%,50%,75%,max,std
Customer ID,"4,266.00","15,348.57","12,346.00","13,885.00","15,348.50","16,833.75","18,287.00","1,698.49"
observation_invoice_count,"4,266.00",4.32,1.00,1.00,2.00,5.00,197.00,7.76
observation_total_value,"4,266.00","1,997.22",2.95,307.54,689.07,"1,676.38","321,329.74","8,601.79"
observation_average_invoice_value,"4,266.00",381.63,2.95,181.67,290.06,424.88,"11,880.84",506.53
observation_median_invoice_value,"4,266.00",364.99,2.95,172.26,287.21,410.38,"11,880.84",477.21
observation_total_quantity,"4,266.00","1,261.13",1.00,157.25,379.00,977.75,"220,600.00","6,403.50"
observation_product_line_count,"4,266.00",92.16,1.00,18.00,43.00,100.00,"5,409.00",195.61
first_purchase_date,4266,2010-04-26 16:28:58.410689,2009-12-01 07:45:00,2010-01-14 11:00:00,2010-03-30 17:09:00,2010-08-09 10:05:00,2010-11-30 18:46:00,NaN
last_purchase_date,4266,2010-09-04 07:55:20.267229,2009-12-01 09:55:00,2010-07-19 13:35:30,2010-10-14 10:05:00,2010-11-15 11:25:45,2010-11-30 19:35:00,NaN
recency_days,"4,266.00",87.22,0.00,15.00,47.00,134.00,364.00,95.11


## 6. Compare candidate future-value target definitions

Candidate A is **future gross purchase value** from qualifying positive purchase lines. Candidate B is **future net transaction value** across identified-customer lines, including negative/return lines and cancelled invoices where customer/date are present.

This comparison is diagnostic. Choose the target definition only after inspecting the results and documenting the intended business meaning.

In [6]:
target_purchases = target_purchases.copy()
target_purchases["line_revenue"] = target_purchases["Quantity"] * target_purchases["Price"]

target_gross = (
    target_purchases.groupby("Customer ID")["line_revenue"]
    .sum()
    .rename("future_gross_purchase_value")
)

target_all = target_all.copy()
target_all["line_revenue"] = target_all["Quantity"] * target_all["Price"]
target_net = (
    target_all.groupby("Customer ID")["line_revenue"]
    .sum()
    .rename("future_net_transaction_value")
)

targets = pd.concat([target_gross, target_net], axis=1).fillna(0).reset_index()
target_summary = targets[
    ["future_gross_purchase_value", "future_net_transaction_value"]
].describe(percentiles=[.5, .75, .9, .95, .99]).T
display(target_summary)

print("Gross target customers with positive value:",
      f"{(targets['future_gross_purchase_value'] > 0).sum():,}")
print("Net target customers with negative value:",
      f"{(targets['future_net_transaction_value'] < 0).sum():,}")
print("Customers with no qualifying future gross purchase:",
      f"{(targets['future_gross_purchase_value'] == 0).sum():,}")

,count,mean,std,min,50%,75%,90%,95%,99%,max
future_gross_purchase_value,"4,331.00","2,009.95","8,633.50",0.00,674.45,"1,623.05","3,614.92","5,902.94","18,831.08","276,006.37"
future_net_transaction_value,"4,331.00","1,907.68","8,257.43","-4,287.63",656.63,"1,582.90","3,506.54","5,732.89","17,126.61","272,787.56"


Gross target customers with positive value: 4,297
Net target customers with negative value: 42
Customers with no qualifying future gross purchase: 34


## 7. Assess customer coverage and target behaviour

In [7]:
feature_ids = set(feature_agg["Customer ID"])
target_ids = set(targets["Customer ID"])

target_review = feature_agg[["Customer ID"]].merge(
    targets, on="Customer ID", how="left", validate="one_to_one"
)
target_review[
    ["future_gross_purchase_value", "future_net_transaction_value"]
] = target_review[
    ["future_gross_purchase_value", "future_net_transaction_value"]
].fillna(0)

coverage_summary = pd.DataFrame({
    "measure": [
        "Customers with observation features",
        "Customers present in target-period transactions",
        "Observation customers with positive future gross value",
        "Observation customers without positive future gross value",
        "Observation customers with no target-period transaction row",
    ],
    "count": [
        len(feature_ids),
        len(target_ids),
        int((target_review["future_gross_purchase_value"] > 0).sum()),
        int((target_review["future_gross_purchase_value"] <= 0).sum()),
        len(feature_ids - target_ids),
    ],
})
display(coverage_summary)

display(
    target_review[
        ["future_gross_purchase_value", "future_net_transaction_value"]
    ].describe(percentiles=[.5, .75, .9, .95, .99]).T
)

,measure,count
0,Customers with observation features,4266
1,Customers present in target-period transactions,4331
2,Observation customers with positive future gro...,2713
3,Observation customers without positive future ...,1553
4,Observation customers with no target-period tr...,1521


,count,mean,std,min,50%,75%,90%,95%,99%,max
future_gross_purchase_value,"4,266.00","1,685.32","8,614.19",0.00,325.01,"1,317.17","3,341.30","5,511.72","17,275.16","276,006.37"
future_net_transaction_value,"4,266.00","1,607.75","8,281.71","-4,287.63",314.57,"1,288.57","3,253.59","5,403.41","16,443.15","272,787.56"


## 8. Assemble a leakage-safe candidate modelling table

The candidate table includes only customers with qualifying purchases in the observation window. Target columns are kept clearly separate from features. Dates used to derive features are removed from the model matrix; the customer identifier is retained for traceability but must not be used as a predictor.

In [8]:
candidate_dataset = feature_agg.merge(
    targets,
    on="Customer ID",
    how="left",
    validate="one_to_one",
)

candidate_dataset[
    ["future_gross_purchase_value", "future_net_transaction_value"]
] = candidate_dataset[
    ["future_gross_purchase_value", "future_net_transaction_value"]
].fillna(0)

candidate_dataset = candidate_dataset.sort_values("Customer ID").reset_index(drop=True)

feature_columns = [
    "observation_invoice_count",
    "observation_total_value",
    "observation_average_invoice_value",
    "observation_median_invoice_value",
    "observation_total_quantity",
    "observation_product_line_count",
    "recency_days",
    "customer_tenure_days",
    "average_days_between_observed_purchases",
]

target_columns = [
    "future_gross_purchase_value",
    "future_net_transaction_value",
]

print("Candidate dataset shape:", candidate_dataset.shape)
print("Feature columns:", feature_columns)
print("Target columns:", target_columns)
print("Customer ID is excluded from feature_columns:", "Customer ID" not in feature_columns)
display(candidate_dataset.head())

Candidate dataset shape: (4266, 14)
Feature columns: ['observation_invoice_count', 'observation_total_value', 'observation_average_invoice_value', 'observation_median_invoice_value', 'observation_total_quantity', 'observation_product_line_count', 'recency_days', 'customer_tenure_days', 'average_days_between_observed_purchases']
Target columns: ['future_gross_purchase_value', 'future_net_transaction_value']
Customer ID is excluded from feature_columns: True


,Customer ID,observation_invoice_count,observation_total_value,observation_average_invoice_value,observation_median_invoice_value,observation_total_quantity,observation_product_line_count,first_purchase_date,last_purchase_date,recency_days,customer_tenure_days,average_days_between_observed_purchases,future_gross_purchase_value,future_net_transaction_value
0,"12,346.00",11,372.86,33.90,22.50,70,33,2009-12-14 08:34:00,2010-06-28 13:53:00,155,196,19.60,"77,183.60",0.00
1,"12,347.00",1,611.53,611.53,611.53,509,40,2010-10-31 14:20:00,2010-10-31 14:20:00,30,0,NaN,"4,796.97","4,796.97"
2,"12,348.00",1,222.16,222.16,222.16,373,20,2010-09-27 14:59:00,2010-09-27 14:59:00,64,0,NaN,"1,797.24","1,797.24"
3,"12,349.00",3,"2,671.14",890.38,"1,068.52",993,102,2010-04-29 13:20:00,2010-10-28 08:23:00,33,181,90.50,"1,757.55","1,757.55"
4,"12,351.00",1,300.93,300.93,300.93,261,21,2010-11-29 15:23:00,2010-11-29 15:23:00,1,0,NaN,0.00,0.00


## 9. Data quality, leakage and target checks

Review these checks before selecting a target or saving the modelling dataset.

In [9]:
print("Missing values in candidate features:")
display(candidate_dataset[feature_columns].isna().sum().to_frame("missing_count"))

print("\nInfinite feature values:")
display(pd.Series({
    col: int(np.isinf(candidate_dataset[col].dropna()).sum())
    for col in feature_columns
}, name="infinite_count").to_frame())

print("\nTarget leakage checklist:")
print("- Observation features end before:", OBS_END)
print("- Target window starts at:", TARGET_START)
print("- Customer ID excluded from model features:", "Customer ID" not in feature_columns)
print("- Target columns excluded from model features:", not set(target_columns).intersection(feature_columns))
print("- Feature date columns excluded from model features:",
      not {"first_purchase_date", "last_purchase_date"}.intersection(feature_columns))

Missing values in candidate features:


,missing_count
observation_invoice_count,0
observation_total_value,0
observation_average_invoice_value,0
observation_median_invoice_value,0
observation_total_quantity,0
observation_product_line_count,0
recency_days,0
customer_tenure_days,0
average_days_between_observed_purchases,1431



Infinite feature values:


,infinite_count
observation_invoice_count,0
observation_total_value,0
observation_average_invoice_value,0
observation_median_invoice_value,0
observation_total_quantity,0
observation_product_line_count,0
recency_days,0
customer_tenure_days,0
average_days_between_observed_purchases,0



Target leakage checklist:
- Observation features end before: 2010-12-01 00:00:00
- Target window starts at: 2010-12-01 00:00:00
- Customer ID excluded from model features: True
- Target columns excluded from model features: True
- Feature date columns excluded from model features: True


## 10. Approved CLV definition and decision log

### Approved candidate design

- **Observation window:** 1 Dec 2009 to 30 Nov 2010 (end-exclusive boundary: 1 Dec 2010).
- **Prediction window:** 1 Dec 2010 to 30 Nov 2011 (end-exclusive boundary: 1 Dec 2011).
- **Target:** `future_gross_purchase_value` — sum of `Quantity × Price` for qualifying positive, non-cancelled purchase lines in the prediction window.
- **Interpretation:** a one-year **future purchase-value proxy**, not accounting profit and not true lifetime value.
- **Zero future value:** retain customers with no qualifying positive purchases in the prediction window and assign target value 0. This is essential to represent non-repeat purchasing.
- **Returns/cancellations:** exclude cancellation invoices and non-positive quantity/price rows from this target. The alternative net-transaction measure remains diagnostic only.
- **Missing Customer IDs:** exclude from customer-level supervised modelling because they cannot be reliably linked to a customer.
- **Duplicates/extreme values:** do not silently delete or cap them. Investigate and document their treatment using training data only where modelling decisions are required.
- **Feature gap:** `average_days_between_observed_purchases` is missing for customers with one observation-period invoice. Preserve it as missing here; handle missingness within the training pipeline in Notebook 03 and consider a single-invoice indicator.
- **Leakage:** only observation-window data may produce predictors. Customer ID, date columns and both target columns must not be predictors.

### Why gross future purchase value?

For this first supervised model, the business question is: **“How much qualifying purchase value is this customer likely to generate in the next year?”** A positive-purchase target is easier to interpret and avoids mixing cancellation/return bookkeeping into the primary target. The net-transaction measure is retained for sensitivity analysis, not as the primary target.

### Final checks before saving

Review the output counts and distributions above. The chosen dates are supported by the dataset coverage, but the dataset ends on 9 Dec 2011; therefore the final 1–9 Dec 2011 records are intentionally outside the one-year target window.

- [x] Candidate observation and prediction windows reviewed.
- [x] Primary target selected and business meaning documented.
- [x] Customers with zero future qualifying purchase value retained.
- [x] Missing-ID, cancellation, return, duplicate and outlier handling documented.
- [x] Feature list and leakage rules documented.

**Next notebook:** `03_modeling_and_evaluation.ipynb`. Save the approved candidate table below, then load it in Notebook 03.


## 11. Save the approved modelling dataset

This exports the candidate table with traceability columns and both target definitions. Notebook 03 must use only the approved feature list and `future_gross_purchase_value` as the primary target.

In [10]:
PROCESSED_DIR = MODEL_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = PROCESSED_DIR / "clv_customer_features_and_targets.csv"
FEATURES_PATH = PROCESSED_DIR / "clv_feature_columns.txt"

candidate_dataset.to_csv(DATASET_PATH, index=False)
FEATURES_PATH.write_text("\\n".join(feature_columns), encoding="utf-8")

print("Saved modelling dataset:", DATASET_PATH)
print("Rows:", f"{len(candidate_dataset):,}")
print("Primary target: future_gross_purchase_value")
print("Features:", len(feature_columns))
print("Target columns retained for audit:", target_columns)


Saved modelling dataset: d:\MCA PU\Semesters\Semester 3\MCA_Projects\the-enterprise-ai-decision-intelligence-platform\ml\model_03_customer_lifetime_value\data\processed\clv_customer_features_and_targets.csv
Rows: 4,266
Primary target: future_gross_purchase_value
Features: 9
Target columns retained for audit: ['future_gross_purchase_value', 'future_net_transaction_value']
